# SNAP-C1 V2: Fractal Recurrent Core (FRC) Pre-Training

This notebook trains the "Hollow Core" of the V2 architecture on purely abstract logic (math, boolean algebra, and algorithmic traces) to teach it foundational reasoning without memorizing human tokens.

**Hardware:** Requires a T4 or A100 GPU instance.

### Step 1: Clone Repository and Setup Environment

In [ ]:
!git clone https://github.com/IRSPlays/SNAP-C1.git
%cd SNAP-C1

# Ensure project root is in python path
import sys
import os
sys.path.append(os.getcwd())

!pip install -q safetensors loguru tqdm

### Step 2: Generate the Abstract Logic Dataset
We don't use HuggingFace datasets. We procedurally synthesize perfect mathematical ground-truth data.

In [ ]:
from v2_core.generate_logic_dataset import LogicDatasetGenerator

print("Forging the logic dataset...")
generator = LogicDatasetGenerator()
# Change this to 1_000_000 for a true pre-training run
NUM_SAMPLES = 100_000 
generator.generate_dataset(num_samples=NUM_SAMPLES, output_file="frc_pretrain_data.jsonl")
print("Logic data successfully synthesized.")

### Step 3: PyTorch Pre-Training Loop
We initialize the recurrent PyTorch core and run a standard backprop loop to align the latent layers.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import json
from tqdm import tqdm
from v2_core.architecture.recurrent_core import FractalRecurrentCore

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

class LogicDataset(Dataset):
    def __init__(self, jsonl_file):
        self.data = []
        with open(jsonl_file, 'r') as f:
            for line in f:
                self.data.append(json.loads(line))
                
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        # For this base implementation, we mock the real embedding projection.
        # In reality, this requires the Hologram compression layer.
        mock_latent_vector = torch.randn(32, 1024) # [seq_len, dim]
        return mock_latent_vector

dataset = LogicDataset("v2_core/data/frc_pretrain_data.jsonl")
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

model = FractalRecurrentCore(hidden_dim=1024, max_loops=12).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Dummy loss function for the hollow core prototype
# In the full system, this connects to the RLFS sandbox outputs
criterion = nn.MSELoss()

epochs = 3
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    
    # Wrap dataloader in tqdm for the progress bar
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch in progress_bar:
        inputs = batch.to(device)  # [batch, seq_len, dim]
        
        optimizer.zero_grad()
        
        # The model loops theoretically independent of batch size
        outputs, loops_taken, confs = model(inputs)
        
        # Self-supervised mock alignment strategy:
        # Teach the recurrent matrices to stabilize the thought vector
        loss = criterion(outputs, inputs)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item(), 'avg_loops': loops_taken})
        
    print(f"Epoch {epoch+1} Complete. Average Loss: {epoch_loss / len(dataloader):.4f}")

print("\n--- CORE PRE-TRAINING COMPLETE ---")
torch.save(model.state_dict(), "frc_pretrained_core.pt")
print("Saved weights to frc_pretrained_core.pt")

try:
    from google.colab import files
    files.download('frc_pretrained_core.pt')
    print("Downloading weights to your PC...")
except ImportError:
    print("Not running in Colab. File saved locally.")